# Saina Helm in Colab

[Saina](https://saina.run) Helm answers **typed questions** about a piece of text or JSON.
Instead of generating free text, it scores every option you give it and returns probabilities,
so your code can branch on the answer — or abstain and hand off when it is unsure.

Run Helm three ways. Every example in this notebook works the same with each one:

| Backend | What you need | Runtime |
|---|---|---|
| **`huggingface`** (default) | nothing — downloads [`run-saina/saina-helm-0.8b`](https://huggingface.co/run-saina/saina-helm-0.8b) and runs it locally with the Saina SDK pipeline | GPU recommended (*Runtime → Change runtime type → T4 GPU*); CPU works, slower |
| **`replicate`** | a `REPLICATE_API_TOKEN` secret; calls [`run-saina/saina-helm-0.8b`](https://replicate.com/run-saina/saina-helm-0.8b) on Replicate | CPU |
| **`endpoint`** | `SAINA_BASE_URL` + `SAINA_API_KEY` secrets for your own Saina server | CPU |

Contents:

1. Install and choose a backend
2. The four question types: `yes_no`, `single_choice`, `rating`, `multi_choice`
3. `distribution` vs `decision` mode: thresholds, margins and abstention
4. Structured (JSON) state and scoring a small batch
5. The low-level SDK pipeline (`huggingface` backend)

## 1. Install

In [ ]:
%pip install -q "saina[local] @ git+https://github.com/run-saina/saina.git" "pydantic>=2.9,<3" replicate

## Choose a backend

Secrets go in Colab's 🔑 **Secrets** panel (left sidebar) with notebook access enabled — never in cells.
Outside Colab, environment variables with the same names work too.

In [ ]:
BACKEND = "huggingface"  # @param ["huggingface", "replicate", "endpoint"]

import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not in Colab, or secret missing / access not granted
        return os.environ.get(name)

HF_REPO = "run-saina/saina-helm-0.8b"
HF_REVISION = "0f24826ab24cf98962d79b6461315f8798d0b5da"  # pinned commit: weights can't change under you
REPLICATE_MODEL = "run-saina/saina-helm-0.8b:c9dd1da66266d604acf37819ae24ea5ad372dca587cb9ef2e006bec391fe9c4f"
print("Backend:", BACKEND)

Every backend speaks the same native request format (the body of `POST /v1/ask`), so we wrap
each one in a single `ask(state, questions, **options)` helper and validate every response
against the shared contract in `saina.contract`.

For `huggingface`, the first run downloads ~1.7 GB of weights (about a minute on Colab).

In [ ]:
import json
from saina import contract

def native_request(state, questions, **options):
    request = dict(model='helm-0.8b', state=state, questions=questions, **options)
    contract.AskRequest.model_validate(request)  # fail fast on malformed questions
    return request

if BACKEND == 'huggingface':
    import torch
    from transformers import pipeline
    from saina import register_transformers

    register_transformers()  # adds the 'saina-classification' pipeline task
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    helm = pipeline('saina-classification', model=HF_REPO, revision=HF_REVISION,
                    device=device, model_kwargs={'device': device})

    def ask(state, questions, **options):
        request = contract.AskRequest.model_validate(native_request(state, questions, **options))
        return contract.ask(request, helm.model.scorer)

elif BACKEND == 'replicate':
    import replicate
    os.environ['REPLICATE_API_TOKEN'] = secret('REPLICATE_API_TOKEN') or ''
    assert os.environ['REPLICATE_API_TOKEN'], 'Add a REPLICATE_API_TOKEN secret'

    def ask(state, questions, **options):
        request = native_request(state, questions, **options)
        # request_json carries the full native request; the form fields are required
        # by the model's input schema but ignored when request_json is present.
        output = replicate.run(REPLICATE_MODEL, input={
            'context': '-', 'question': '-', 'answers': 'a\nb',
            'request_json': json.dumps(request)})
        return contract.AskResponse.model_validate(output).model_dump(exclude_unset=True)

elif BACKEND == 'endpoint':
    from saina import Saina
    client = Saina(secret('SAINA_BASE_URL'), api_key=secret('SAINA_API_KEY'))

    def ask(state, questions, **options):
        # The HTTP client validates the response against your request before returning.
        return client.ask(model='helm-0.8b', state=state, questions=questions, **options)

else:
    raise ValueError(f'Unknown backend: {BACKEND}')

print('Ready:', BACKEND, '| device: ' + device if BACKEND == 'huggingface' else '')

## 2. Question types

A request has one `state` (the text or JSON being judged) and up to 256 named `questions`,
all answered about that same state.

In [ ]:
ticket = ("Hi, I was charged twice for my order #4821 yesterday. "
          "Please refund the duplicate charge and tell me why this happened.")

result = ask(ticket, {
    'is_billing': {
        'type': 'yes_no',
        'question': 'Is this message about billing or payments?',
    },
    'issue': {
        'type': 'single_choice',
        'question': 'What is the customer\'s main issue?',
        'options': {
            'duplicate_charge': 'Charged more than once',
            'card_delivery': 'Card has not arrived',
            'login': 'Cannot log in',
            'other': 'Something else',
        },
    },
    'urgency': {
        'type': 'rating',
        'question': 'How urgent is this request?',
        'levels': ['not urgent', 'normal', 'urgent', 'emergency'],
    },
    'wants': {
        'type': 'multi_choice',
        'question': 'What does the customer ask for? Select all that apply.',
        'options': {
            'refund': 'A refund',
            'explanation': 'An explanation of what happened',
            'cancel': 'Cancel the order',
        },
    },
})

print(json.dumps(result, indent=2))

What came back:

- **`yes_no`** → `yes` / `no` probabilities and a `confidence` in [0, 1].
- **`single_choice`** → a `probabilities` distribution over your option keys plus the top `selection`.
- **`rating`** → a distribution over level indices, the `expected_level`, and an ordinal-aware `confidence`.
- **`multi_choice`** → independent `memberships` per option (they do *not* sum to 1).
- **`usage`** → input tokens; `output_tokens` is always 0 because Helm never generates text.

Option values are optional descriptions: `{'refund': None}` uses the key alone.

## 3. Decision mode: let the model abstain

`distribution` (the default) always returns probabilities. `decision` also applies a policy
and tells you whether to act:

- `threshold` — the top probability must reach this (default 0.8)
- `min_margin` — the gap between the top two must reach this (`yes_no` / `single_choice`)
- `reason` — one of `accepted`, `below_threshold`, `below_margin`, `tie`

When the policy is not met, `selection` / `selected` / `level` is `None`: route to a human
instead of guessing. Policies can be set per request and overridden per question.

In [ ]:
route = {
    'type': 'single_choice',
    'question': 'Which team should handle this message?',
    'options': {
        'billing': 'Billing: charges, refunds, payments',
        'shipping': 'Shipping: delivery and tracking',
        'security': 'Security: fraud, stolen cards, account takeover',
    },
    'min_margin': 0.3,  # per-question override
}

messages = [
    ticket,
    "My package tracking hasn't updated in a week.",
    "Someone I don't know just logged into my account from another country.",
    "There's a charge I don't recognise and my parcel never arrived either.",
    "Do you have a careers page?",
]

for message in messages:
    answer = ask(message, {'route': route}, mode='decision', threshold=0.9)['answers']['route']
    action = f"→ {answer['selection']}" if answer['selection'] else '→ human review'
    top = max(answer['probabilities'].values())
    print(f"{answer['reason']:>16}  p={top:.2f}  {action:<16} {message[:55]}")

The careers question fits none of the three teams, so the model is forced to spread or lower
its probability — and the policy turns that into an abstention. Tune `threshold` and
`min_margin` on your own labelled examples to trade automation rate against error rate.

## 4. Structured state and batches

`state`, `question` and option descriptions accept JSON objects as well as strings, so you can
pass records straight from a database or an agent's tool output.

In [ ]:
import pandas as pd

orders = [
    {'order_id': 4821, 'status': 'delivered', 'customer_note': 'Box arrived crushed and the screen is cracked.'},
    {'order_id': 4822, 'status': 'in_transit', 'customer_note': 'Any idea when this will arrive?'},
    {'order_id': 4823, 'status': 'delivered', 'customer_note': 'Works perfectly, thanks!'},
]

questions = {
    'needs_action': {'type': 'yes_no', 'question': 'Does this order need follow-up from support?'},
    'sentiment': {'type': 'rating', 'question': 'How does the customer feel?',
                  'levels': ['very unhappy', 'neutral', 'very happy']},
}

rows = []
for order in orders:
    answers = ask(order, questions)['answers']
    rows.append({
        'order_id': order['order_id'],
        'needs_action_p': round(answers['needs_action']['yes'], 3),
        'sentiment (0-2)': round(answers['sentiment']['expected_level'], 2),
        'sentiment_conf': round(answers['sentiment']['confidence'], 3),
    })

pd.DataFrame(rows)

## 5. The low-level SDK pipeline

With the `huggingface` backend, `helm` is a regular 🤗 Transformers pipeline registered by the
Saina SDK. It takes a plain `task`, `input` and list of `choices`, and returns probabilities
in your choice order — handy when you don't need the typed-question layer.

```python
from saina import register_transformers
from transformers import pipeline

register_transformers()
helm = pipeline('saina-classification', model='run-saina/saina-helm-0.8b',
                revision='<commit>', device='cuda:0', model_kwargs={'device': 'cuda:0'})
```

In [ ]:
if BACKEND != 'huggingface':
    print('Switch BACKEND to "huggingface" to run the local pipeline.')
else:
    print(helm({
        'task': 'Identify the banking issue.',
        'input': 'I was charged twice.',
        'choices': ['duplicate charge', 'card delivery'],
        'mode': 'single_label',   # probabilities sum to 1
    }))
    print(helm({
        'task': 'Which topics apply?',
        'input': 'I was charged twice. Please refund the duplicate charge.',
        'choices': ['Billing or charges', 'Refund request', 'Delivery status'],
        'mode': 'multi_label',    # independent memberships
    }))

## Next steps

- SDK source, JavaScript client and self-hosting guide: [github.com/run-saina/saina](https://github.com/run-saina/saina)
- Model: [Hugging Face](https://huggingface.co/run-saina/saina-helm-0.8b) · [Replicate](https://replicate.com/run-saina/saina-helm-0.8b)
- n8n integration: [github.com/run-saina/n8n-nodes](https://github.com/run-saina/n8n-nodes)
- Integration help: [dev@saina.run](mailto:dev@saina.run)